# Episode 14: Give Your Agent a UI

Your agent works great in a terminal. But your users don't live in terminals.

**By the end of this episode, you will:**

- Build a complete chat UI with Streamlit in about 20 lines of Python
- Understand how AG-UI's event protocol connects any agent backend to any frontend
- Know when to reach for each approach — and what you trade off in doing so

> This episode has a companion code folder at `ep14_agent_ui/`.

## Why Your Agents Need a UI

Nobody outside your team is going to `pip install` anything, open a terminal, and type Python to talk to your agent. If you want real users — customers, colleagues, stakeholders — you need an interface they already understand: a chat window in a browser.

Gartner projects that 40% of enterprise applications will embed agentic AI by 2026. Most of those will need a UI layer, and the options range from "ship a demo by lunch" to "production-grade protocol for any frontend framework."

| Approach | Description | Best For |
|----------|-------------|----------|
| **Streamlit** | Python-only web apps | Quick prototyping, demos |
| **AG-UI Protocol** | Standardized SSE event stream | Production apps, any frontend |
| **Gradio** | ML-focused web UIs | Model demos, data science |
| **CopilotKit** | React-based copilot framework | Embedded AI assistants |

We will focus on two: Streamlit (the fastest path from agent to web app) and AG-UI (the protocol built for production).

## Part 1: Streamlit — The Fastest Path from Agent to Web App

Streamlit turns a single Python file into a running web application. No HTML, no JavaScript, no build step. For internal tools, demos, and prototypes, it is hard to beat.

**Important:** Streamlit apps don't run inside Jupyter notebooks. Save the code below to a `.py` file and launch it from the command line.

```bash
pip install streamlit
streamlit run ep14_agent_ui/streamlit_app.py
```

In [ ]:
# Save this as ep14_agent_ui/streamlit_app.py and run: streamlit run ep14_agent_ui/streamlit_app.py

import streamlit as st
from dotenv import load_dotenv

load_dotenv()
from autogen import AssistantAgent, LLMConfig

st.title("AG2 Chat Assistant")
llm_config = LLMConfig({"model": "gpt-4o-mini"})

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Ask me anything"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    user = AssistantAgent("user", human_input_mode="NEVER")
    agent = AssistantAgent(
        "assistant", system_message="You are helpful.", llm_config=llm_config
    )

    result = user.run(agent, message=prompt, max_turns=1)
    result.process()
    response = result.summary

    st.session_state.messages.append({"role": "assistant", "content": response})
    with st.chat_message("assistant"):
        st.markdown(response)

### How It Works

Streamlit re-runs the entire script on every user interaction, which sounds wasteful until you realize it makes the programming model dead simple.

`st.session_state.messages` persists the chat history across those reruns. `st.chat_message` renders each message with the right avatar. `st.chat_input` provides the text box at the bottom. And `agent.run()` sends the user's input to your AG2 agent and returns the response.

That's a complete chat UI in roughly 20 lines of Python — no frontend code at all.

## Part 2: AG-UI — The Production Protocol

For production, you want a standard protocol — not a framework-specific hack. **AG-UI** defines a clean event stream between your agent backend and any frontend, using Server-Sent Events over HTTP.

### How AG-UI Works

```
┌─────────────┐     SSE Events      ┌─────────────┐
│  FastAPI     │ ──────────────────> │  Frontend    │
│  Backend     │                     │  (any JS     │
│  (AG2 agent) │ <────────────────── │   framework) │
└─────────────┘     HTTP requests    └─────────────┘
```

The backend streams events to the frontend in real time. The event vocabulary is intentionally small and covers the full agent lifecycle:

- **`TEXT_MESSAGE_START`** / **`TEXT_MESSAGE_CONTENT`** / **`TEXT_MESSAGE_END`** — streaming text tokens
- **`TOOL_CALL_START`** / **`TOOL_CALL_END`** — tool invocations
- **`STATE_SNAPSHOT`** — full state for multi-stage pipelines
- **`RUN_STARTED`** / **`RUN_FINISHED`** — lifecycle bookends

Because the protocol is framework-agnostic, your frontend team can use React, Vue, Svelte, or plain JavaScript — whatever they prefer. The **AG2 Playground** at [playground.ag2.ai](https://playground.ag2.ai) is built entirely on AG-UI.

### AG-UI Examples in This Repo

The companion folder includes two working examples that show the protocol at different scales.

**`ag-ui/weather/`** is the minimal case — a single weather agent with tool calls, a FastAPI backend streaming SSE events, and a lightweight HTML/JS frontend consuming the stream.

**`ag-ui/factory/`** shows a multi-stage pipeline where several agents hand off in sequence (researcher, analyst, writer). It uses `STATE_SNAPSHOT` events to let the frontend display which stage is currently active.

Both follow the same launch pattern:

```bash
# Terminal 1: Start the backend
cd ag-ui/weather && uvicorn server:app --port 8000

# Terminal 2: Open the frontend
open http://localhost:8000/static/index.html
```

> No runnable AG-UI code in this notebook — see the companion examples for full implementations.

### CopilotKit Integration

If your team already works in React, **CopilotKit** offers a polished copilot experience with AG2 on the backend. It handles chat UI components, streaming responses, action suggestions, and shared state between the frontend and agent — all out of the box.

See the AG2 blog for integration guides.

## Choosing Between Streamlit and AG-UI

Both approaches get your agent into a browser, but they optimize for different things. Streamlit prioritizes speed-to-demo; AG-UI prioritizes flexibility and production readiness.

| Aspect | Streamlit | AG-UI |
|--------|-----------|-------|
| **Setup** | 1 file, `pip install streamlit` | FastAPI + frontend |
| **Time to first demo** | Minutes | Hours |
| **Streaming** | Limited | Full SSE streaming |
| **Frontend flexibility** | Python-only | Any JS framework |
| **Multi-agent visibility** | Basic | Full pipeline state |
| **Best for** | Demos, internal tools | User-facing products |

A common pattern in practice: prototype with Streamlit to validate the agent logic, then migrate to AG-UI when you are ready to ship to real users.

## Try It Yourself

Modify the Streamlit app to show **which agent is speaking**. When you have multiple agents in a conversation, display the agent's name above each response.

Hint: wrap the response rendering like this:
```python
with st.chat_message("assistant"):
    st.caption(f"Agent: {agent_name}")
    st.markdown(response)
```

## What's Next

In **Episode 15**, you will connect your agents to external tools via **MCP** (Model Context Protocol) — the universal standard for agent-tool communication.